<a href="https://colab.research.google.com/github/MuhammadQasimTanveer/HerdOrbit-ML/blob/main/symptoms_disease.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Symptoms Disease Prediction using XGBoost

In [21]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

print("Basic Setup completed")

Basic Setup completed


Add dataset loading and basic setup

In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
import joblib

# Load dataset
df = pd.read_excel('symptoms.xlsx')
print(df.head())

    Animal  Age  Temperature         Symptom 1         Symptom 2  \
0      cow    3        103.1        depression    painless lumps   
1  buffalo   13        104.5    painless lumps  loss of appetite   
2    sheep    1        100.5        depression    painless lumps   
3      cow   14        100.3  loss of appetite  swelling in limb   
4    sheep    2        103.6    painless lumps  loss of appetite   

          Symptom 3      Disease  
0  loss of appetite    pneumonia  
1        depression  lumpy virus  
2  loss of appetite  lumpy virus  
3   crackling sound     blackleg  
4        depression    pneumonia  


Add data preprocessing and symptom combination

In [23]:
df['All_Symptoms'] = df['Symptom 1'].astype(str) + " " + df['Symptom 2'].astype(str) + " " + df['Symptom 3'].astype(str)
df = df.drop(['Symptom 1', 'Symptom 2', 'Symptom 3'], axis=1)
print(df.head())

    Animal  Age  Temperature      Disease  \
0      cow    3        103.1    pneumonia   
1  buffalo   13        104.5  lumpy virus   
2    sheep    1        100.5  lumpy virus   
3      cow   14        100.3     blackleg   
4    sheep    2        103.6    pneumonia   

                                        All_Symptoms  
0         depression painless lumps loss of appetite  
1         painless lumps loss of appetite depression  
2         depression painless lumps loss of appetite  
3  loss of appetite swelling in limb crackling sound  
4         painless lumps loss of appetite depression  


Add label encoding for categorical features

In [24]:
# Ye cell X banane se PEHLE chalayein
from sklearn.preprocessing import LabelEncoder

le_animal = LabelEncoder()
le_symptoms = LabelEncoder()
le_disease = LabelEncoder()

# df pe encoding karein
df['Animal'] = le_animal.fit_transform(df['Animal'].astype(str))
df['All_Symptoms'] = le_symptoms.fit_transform(df['All_Symptoms'].astype(str))
df['Disease'] = le_disease.fit_transform(df['Disease'].astype(str))

print("✅ All columns encoded!")
print(df.dtypes)

✅ All columns encoded!
Animal            int64
Age               int64
Temperature     float64
Disease           int64
All_Symptoms      int64
dtype: object


Add train-test split and feature separation

In [25]:
X = df.drop('Disease', axis=1)
y = le_disease.fit_transform(df['Disease'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

 Add XGBoost model training

In [26]:
from sklearn.preprocessing import LabelEncoder

# X ke object columns ko encode karein
for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

print("✅ X encoded! Now training model...")

# Model dobara train karein
model = XGBClassifier(eval_metric='mlogloss', random_state=42)
model.fit(X_train, y_train)

print("✅ Model trained successfully!")

✅ X encoded! Now training model...
✅ Model trained successfully!


In [20]:
print("Data types of X:")
print(X.dtypes)

Data types of X:
Animal           object
Age               int64
Temperature     float64
All_Symptoms     object
dtype: object


Add model evaluation and accuracy report

In [29]:
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(classification_report(y_test, y_pred))

Accuracy: 85.49%
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1596
           1       1.00      1.00      1.00      1425
           2       1.00      1.00      1.00      1381
           3       0.99      0.99      0.99      1992
           4       0.94      0.93      0.94      2016
           5       0.94      0.95      0.95      1922
           6       0.40      0.39      0.40      1382
           7       0.42      0.43      0.43      1425

    accuracy                           0.85     13139
   macro avg       0.84      0.84      0.84     13139
weighted avg       0.85      0.85      0.85     13139

